# 11 — Aim 2 ALT Signature Score (DepMap OS RNA-seq)

Trains a **regularized ALT signature score** on DepMap osteosarcoma RNA-seq using
**direction-aware** labels (`alt_call_direction`), per `label_spec_v1.0` §2. This is a
signature score evaluated by **leave-one-out cross-validation**, *not* a calibrated
binary classifier and *not* a fixed-threshold predictor.

**Scope (do-not-cross lines):** DepMap_26Q1 only (notebook 12 handles other cohorts);
never pool with de Nonneville microarray; no calibrated probabilities / fixed thresholds;
the manifest and all earlier notebooks are read-only; **no preprocessing is ever fit
outside the CV training fold**.

**Two definition runs in parallel**
| Definition | `alt_status` admitted | Expected positive / negative |
|---|---|---|
| Primary (strict) | `alt_gold_positive`, `alt_negative_high_conf` | 12 / 8 (N=20) |
| Pragmatic (sensitivity) | + `alt_silver` | 18 / 13 (N=31) |

**Arms come from `alt_call_direction`, never `alt_status`** — using `alt_status` would
corrupt the silver split. Verified by a per-row assertion before any CV.

**Model:** `LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
max_iter=10000, class_weight='balanced', random_state=42)` inside a
`Pipeline(VarianceGeneFilter → StandardScaler → LogisticRegression)`.

**CV:** outer LOOCV; inside each fold a `StandardScaler` + a training-fold variance gene
filter + inner 5-fold CV to select `C` from `10**linspace(-2,2,9)`; refit on the N-1
fold; predict the held-out sample. AUROC / AUPRC / sensitivity@90%spec with 1000-bootstrap
95% CIs; a 100-shuffle label-permutation null.

**Outputs (`data/processed/`):** `notebook_11_report.md`, per-definition prediction CSVs,
ranked gene CSVs, and refit `Pipeline` `.joblib` files. Re-running is **byte-identical**
(deterministic seeds + CV iterators; sha256-verified).

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

# All random seeds set to 42 (spec).
random.seed(42)
np.random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

# penalty='elasticnet' is deprecated in sklearn 1.8 but mandated by the Aim 2 model
# spec; silence the expected FutureWarning so notebook output stays clean.
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.utils import log_versions, sha256_file
import pandas as pd

log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'joblib'])
print(f'Run: {RUN_TIMESTAMP}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  sklearn: 1.8.0
  joblib: 1.5.3
Run: 2026-05-31T03:55:25Z


## 1 — Load manifest and DepMap RNA membership (cheap probe, no matrix load)

In [2]:
from src.aim2_signature import (
    apply_inclusion_filter, assert_expected_counts, assert_ground_truth,
    direction_counts, labels_from_direction, load_expression,
    run_loocv, evaluate, bootstrap_ci, permutation_test, refit_full,
    ranked_genes, top_genes, flag_alt_genes, compare_definitions,
    EXPECTED_COUNTS, GROUND_TRUTH_DIRECTION, ALT_GENE_LIST, C_GRID,
    TOP_K_VARIABLE_GENES, N_PERMUTATIONS, N_BOOTSTRAP, RANDOM_SEED, L1_RATIO,
    PRIMARY_ALT_STATUSES, PRAGMATIC_ALT_STATUSES,
)
from src.aim1_inventory import DEPMAP_RNA_PATH, DEPMAP_RNA_DATA_TYPE, depmap_rna_model_ids

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
print(f'Manifest rows: {len(manifest)}')

# Membership probe reads only the ModelID column — the full matrix is NOT loaded yet.
depmap_models = depmap_rna_model_ids(DEPMAP_RNA_PATH)
print(f'DepMap RNA ModelIDs (ModelID-column probe only): {len(depmap_models)}')
print(f'RNA matrix: {DEPMAP_RNA_PATH.relative_to(ROOT)}  [{DEPMAP_RNA_DATA_TYPE}]')
print(f'Config: top_k={TOP_K_VARIABLE_GENES}, C_grid={[round(c,4) for c in C_GRID]}, '
      f'l1_ratio={L1_RATIO}, n_perm={N_PERMUTATIONS}, n_boot={N_BOOTSTRAP}, seed={RANDOM_SEED}')

Manifest rows: 115
DepMap RNA ModelIDs (ModelID-column probe only): 1719
RNA matrix: data/raw/depmap/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv  [log2(TPM+1) protein-coding]
Config: top_k=500, C_grid=[np.float64(0.01), np.float64(0.0316), np.float64(0.1), np.float64(0.3162), np.float64(1.0), np.float64(3.1623), np.float64(10.0), np.float64(31.6228), np.float64(100.0)], l1_ratio=0.5, n_perm=100, n_boot=1000, seed=42


## 2 — Inclusion filter + filter trace + hard count / ground-truth assertions

Filters are applied **explicitly** (we do not trust `include_*` flags): cohort →
in-RNA-matrix → `alt_call_direction in {positive, negative}` → definition `alt_status`
set. Counts are asserted against the expected 12/8 (primary) and 18/13 (pragmatic);
a mismatch raises `ValueError` with the exact discrepancy.

In [3]:
DEFINITIONS = ['primary', 'pragmatic']
filtered, traces = {}, {}
for defn in DEFINITIONS:
    df, trace = apply_inclusion_filter(manifest, depmap_models, defn)
    filtered[defn], traces[defn] = df, trace
    print(f'=== {defn.upper()} filter trace ===')
    for line in trace.as_lines():
        print(line)
    print(f'  -> direction counts: {direction_counts(df)}')
    print()

# Hard count assertions (raise ValueError with the exact discrepancy on mismatch).
for defn in DEFINITIONS:
    assert_expected_counts(filtered[defn], defn)
print('Count assertions PASSED:', {d: direction_counts(filtered[d]) for d in DEFINITIONS})

=== PRIMARY filter trace ===
  manifest rows (all cohorts)                             N = 115
  cohort == 'DepMap_26Q1'                                 N = 35
  sample_id present in DepMap RNA ModelID index           N = 34
  alt_call_direction in ('positive', 'negative')          N = 31
  alt_status in ('alt_gold_positive', 'alt_negative_high_conf') N = 20
  -> direction counts: {'positive': 12, 'negative': 8}

=== PRAGMATIC filter trace ===
  manifest rows (all cohorts)                             N = 115
  cohort == 'DepMap_26Q1'                                 N = 35
  sample_id present in DepMap RNA ModelID index           N = 34
  alt_call_direction in ('positive', 'negative')          N = 31
  alt_status in ('alt_gold_positive', 'alt_negative_high_conf', 'alt_silver') N = 31
  -> direction counts: {'positive': 18, 'negative': 13}

Count assertions PASSED: {'primary': {'positive': 12, 'negative': 8}, 'pragmatic': {'positive': 18, 'negative': 13}}


### 2a — Arms are derived from `alt_call_direction`, not `alt_status`

The crosstab below shows the silver tier splitting into *both* arms by direction — the
exact split that `alt_status` would destroy. The per-row + anchor assertions abort
before any modelling if any included row lacks a valid direction, or if U2OS
(`ACH-000364`) is not `positive` / SJSA1 (`ACH-000748`) is not `negative`.

In [4]:
print('Pragmatic: alt_status x alt_call_direction (ARMS COME FROM DIRECTION):')
print(pd.crosstab(filtered['pragmatic']['alt_status'],
                  filtered['pragmatic']['alt_call_direction']).to_string())
print()

# Per-row direction + ground-truth anchors — abort before any CV.
for defn in DEFINITIONS:
    assert_ground_truth(filtered[defn])

for sid, exp in GROUND_TRUTH_DIRECTION.items():
    got = filtered['pragmatic'].set_index('sample_id').loc[sid, 'alt_call_direction']
    print(f'  ground-truth {sid}: alt_call_direction={got!r} (expected {exp!r}) '
          f'[{"PASS" if got == exp else "FAIL"}]')
print('Ground-truth + per-row direction assertions PASSED.')

Pragmatic: alt_status x alt_call_direction (ARMS COME FROM DIRECTION):
alt_call_direction      negative  positive
alt_status                                
alt_gold_positive              0        12
alt_negative_high_conf         8         0
alt_silver                     5         6

  ground-truth ACH-000364: alt_call_direction='positive' (expected 'positive') [PASS]
  ground-truth ACH-000748: alt_call_direction='negative' (expected 'negative') [PASS]
Ground-truth + per-row direction assertions PASSED.


## 3 — Load expression for included rows only

The filter is locked, so we now load **only the included rows** from the DepMap matrix
(union = the pragmatic set; primary is a subset). **No preprocessing happens here** —
scaling and variance filtering occur strictly inside each CV training fold.

In [5]:
all_ids = filtered['pragmatic']['sample_id'].tolist()
expr_all = load_expression(all_ids, DEPMAP_RNA_PATH)
gene_columns = list(expr_all.columns)
print(f'Loaded expression: {expr_all.shape[0]} samples x {expr_all.shape[1]} genes '
      f'[{DEPMAP_RNA_DATA_TYPE}]')

data = {}
for defn in DEFINITIONS:
    ids = filtered[defn]['sample_id'].tolist()
    X = expr_all.loc[ids].to_numpy()
    y = labels_from_direction(filtered[defn])
    assert X.shape[0] == len(y) == len(ids)
    data[defn] = {'ids': ids, 'X': X, 'y': y}
    print(f'  {defn}: X={X.shape}  positives={int(y.sum())}  negatives={int((1 - y).sum())}')

Loaded expression: 31 samples x 19215 genes [log2(TPM+1) protein-coding]
  primary: X=(20, 19215)  positives=12  negatives=8
  pragmatic: X=(31, 19215)  positives=18  negatives=13


## 4 — Nested leave-one-out cross-validation

Outer LOOCV. Within each fold: variance gene filter + `StandardScaler` fit on the N-1
training fold only; inner 5-fold CV selects `C`; the inner-selected model is refit on the
full N-1 fold; the held-out sample is scored. `Pipeline` guarantees every preprocessing
step respects fold boundaries.

This is the compute-heavy step (inner C-selection in every outer fold). Together
with the permutation null (§6), expect this notebook to take roughly **10–15 minutes**
on a laptop; both definitions and the 100-shuffle null print progress as they run.

In [6]:
loocv = {}
for defn in DEFINITIONS:
    d = data[defn]
    print(f'LOOCV {defn}: N={len(d["y"])}, top_k={TOP_K_VARIABLE_GENES} ...', flush=True)
    res = run_loocv(d['X'], d['y'], d['ids'], top_k=TOP_K_VARIABLE_GENES,
                    random_state=RANDOM_SEED)
    loocv[defn] = res
    print(f'  AUROC={res.auroc:.4f}  median C={np.median(res.selected_Cs):.4g}  '
          f'C range=[{min(res.selected_Cs):.4g}, {max(res.selected_Cs):.4g}]', flush=True)

LOOCV primary: N=20, top_k=500 ...
  AUROC=0.4688  median C=100  C range=[0.01, 100]
LOOCV pragmatic: N=31, top_k=500 ...
  AUROC=0.6880  median C=31.62  C range=[0.1, 100]


## 5 — Performance with bootstrap 95% CIs

AUROC, AUPRC, and sensitivity at 90% specificity on the aggregated LOOCV predictions; 1000-resample percentile bootstrap CIs.

In [7]:
metrics, cis = {}, {}
for defn in DEFINITIONS:
    res = loocv[defn]
    metrics[defn] = evaluate(res.y_true, res.y_prob)
    cis[defn] = bootstrap_ci(res.y_true, res.y_prob, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)
    m, c = metrics[defn], cis[defn]
    print(f'=== {defn.upper()} (N={len(res.y_true)}) ===')
    print(f"  AUROC        = {m['auroc']:.4f}  95% CI [{c['auroc'][0]:.4f}, {c['auroc'][1]:.4f}]")
    print(f"  AUPRC        = {m['auprc']:.4f}  95% CI [{c['auprc'][0]:.4f}, {c['auprc'][1]:.4f}]")
    print(f"  Sens@90%Spec = {m['sensitivity_at_90_spec']:.4f}  "
          f"95% CI [{c['sensitivity_at_90_spec'][0]:.4f}, {c['sensitivity_at_90_spec'][1]:.4f}]")
    print()

=== PRIMARY (N=20) ===
  AUROC        = 0.4688  95% CI [0.2031, 0.7500]
  AUPRC        = 0.6422  95% CI [0.4300, 0.8879]
  Sens@90%Spec = 0.0833  95% CI [0.0000, 0.5000]

=== PRAGMATIC (N=31) ===
  AUROC        = 0.6880  95% CI [0.4473, 0.8950]
  AUPRC        = 0.7024  95% CI [0.4856, 0.9225]
  Sens@90%Spec = 0.1111  95% CI [0.0000, 0.5789]



## 6 — Permutation null (the honest small-N test)

Labels are shuffled `N_PERMUTATIONS` times within the included samples and the **entire
fold-local LOOCV pipeline is re-run** per shuffle; the empirical p-value is the fraction
of permutations with AUROC ≥ observed.

**Documented deviation:** re-selecting `C` via inner CV inside every fold of every shuffle
is computationally intractable here (~9 h per definition over ~19k genes). The null
therefore fixes `C` to the **median observed LOOCV `C`** and re-runs everything else
fold-locally per shuffle. This is standard for permutation nulls and, if anything,
slightly conservative (permuted labels cannot re-tune `C`).

In [8]:
perms = {}
for defn in DEFINITIONS:
    res = loocv[defn]
    median_C = float(np.median(res.selected_Cs))
    d = data[defn]
    print(f'Permutation null {defn}: {N_PERMUTATIONS} shuffles, fixed C={median_C:.4g} ...',
          flush=True)
    perms[defn] = permutation_test(
        d['X'], d['y'], d['ids'], observed_auroc=res.auroc,
        n_perm=N_PERMUTATIONS, fixed_C=median_C,
        top_k=TOP_K_VARIABLE_GENES, random_state=RANDOM_SEED, seed=RANDOM_SEED,
        progress_every=10,
    )
    p = perms[defn]
    print(f"  observed AUROC={p['observed_auroc']:.4f}  null mean={p['perm_aurocs'].mean():.4f}"
          f"  null 95th pct={np.quantile(p['perm_aurocs'], 0.95):.4f}"
          f"  empirical p={p['p_value']:.4f}", flush=True)

Permutation null primary: 100 shuffles, fixed C=100 ...
    permutation 10/100  (16s elapsed)
    permutation 20/100  (33s elapsed)
    permutation 30/100  (51s elapsed)
    permutation 40/100  (67s elapsed)
    permutation 50/100  (84s elapsed)
    permutation 60/100  (101s elapsed)
    permutation 70/100  (119s elapsed)
    permutation 80/100  (136s elapsed)
    permutation 90/100  (153s elapsed)
    permutation 100/100  (170s elapsed)
  observed AUROC=0.4688  null mean=0.5114  null 95th pct=0.7917  empirical p=0.6000
Permutation null pragmatic: 100 shuffles, fixed C=31.62 ...
    permutation 10/100  (66s elapsed)
    permutation 20/100  (135s elapsed)
    permutation 30/100  (201s elapsed)
    permutation 40/100  (266s elapsed)
    permutation 50/100  (334s elapsed)
    permutation 60/100  (403s elapsed)
    permutation 70/100  (473s elapsed)
    permutation 80/100  (536s elapsed)
    permutation 90/100  (605s elapsed)
    permutation 100/100  (675s elapsed)
  observed AUROC=0.6880 

## 7 — Feature importance (refit on full set)

One final `Pipeline` is fit on all N samples at the **median LOOCV-selected `C`**. The
top-20 genes by |coefficient| (signed) are extracted and cross-referenced against the
curated ALT-related gene list.

In [9]:
final_models, ranked, tops = {}, {}, {}
for defn in DEFINITIONS:
    res = loocv[defn]
    median_C = float(np.median(res.selected_Cs))
    d = data[defn]
    pipe = refit_full(d['X'], d['y'], C=median_C, top_k=TOP_K_VARIABLE_GENES,
                      random_state=RANDOM_SEED)
    final_models[defn] = pipe
    ranked[defn] = ranked_genes(pipe, gene_columns)
    tops[defn] = flag_alt_genes(top_genes(ranked[defn], 20), ALT_GENE_LIST)
    n_nonzero = int((ranked[defn]['coefficient'] != 0).sum())
    print(f'=== {defn.upper()} top-20 genes (refit on full N, C={median_C:.4g}) ===')
    print(f'  non-zero coefficients: {n_nonzero} / {len(ranked[defn])} genes')
    print(tops[defn][['rank', 'gene_symbol', 'coefficient', 'in_alt_gene_list']]
          .to_string(index=False))
    hits = tops[defn].loc[tops[defn]['in_alt_gene_list'], 'gene_symbol'].tolist()
    print(f'  ALT-gene-list hits in top-20: {hits if hits else "none"}')
    print()

=== PRIMARY top-20 genes (refit on full N, C=100) ===
  non-zero coefficients: 405 / 500 genes
 rank gene_symbol  coefficient  in_alt_gene_list
    1        GPX3     0.237479             False
    2      RPL39L    -0.222975             False
    3       CRYAB     0.213287             False
    4      XAGE1B     0.200183             False
    5      MAGEB2     0.183950             False
    6      AMIGO2     0.182823             False
    7     COLEC12    -0.181824             False
    8        LY6E    -0.177242             False
    9      XAGE1A     0.176727             False
   10      IFITM2    -0.174802             False
   11     MAGEA10     0.174662             False
   12      KRBOX1     0.173082             False
   13        BMP4     0.170874             False
   14        TDO2     0.170070             False
   15       PLIN2    -0.165703             False
   16         GAL    -0.165156             False
   17       USP9Y    -0.159018             False
   18       DDX3Y    -0

## 8 — Primary vs pragmatic comparison

AUROC delta, top-20 gene overlap (intersection count + Jaccard), and Pearson correlation of signed coefficients over the shared gene space.

In [10]:
comparison = compare_definitions(metrics['primary'], metrics['pragmatic'],
                                 ranked['primary'], ranked['pragmatic'], n=20)
print('=== Primary vs Pragmatic ===')
print(f"  AUROC delta (primary - pragmatic): {comparison['auroc_delta_primary_minus_pragmatic']:+.4f}")
print(f"  Top-20 overlap: {comparison['top_overlap_count']} genes "
      f"(Jaccard={comparison['top_overlap_jaccard']:.3f})")
print(f"  Overlap genes: {comparison['top_overlap_genes']}")
print(f"  Coefficient Pearson r over {comparison['n_shared_genes']} shared genes: "
      f"{comparison['coef_pearson_on_shared_genes']:.4f}")

=== Primary vs Pragmatic ===
  AUROC delta (primary - pragmatic): -0.2193
  Top-20 overlap: 5 genes (Jaccard=0.143)
  Overlap genes: ['AMIGO2', 'CRYAB', 'GAL', 'GPX3', 'RPL39L']
  Coefficient Pearson r over 416 shared genes: 0.7661


## 9 — Write prediction / gene / model outputs

In [11]:
import joblib

outputs = {}
for defn in DEFINITIONS:
    res = loocv[defn]
    pred_df = pd.DataFrame({
        'sample_id': res.sample_ids,
        'true_label': res.y_true,
        'predicted_prob': res.y_prob,
    })
    pred_path = PROC / f'aim2_signature_{defn}_predictions.csv'
    pred_df.to_csv(pred_path, index=False)
    outputs[pred_path.name] = pred_path

    genes_path = PROC / f'aim2_signature_{defn}_genes.csv'
    ranked[defn].to_csv(genes_path, index=False)
    outputs[genes_path.name] = genes_path

    model_path = PROC / f'aim2_signature_{defn}_model.joblib'
    joblib.dump(final_models[defn], model_path)
    outputs[model_path.name] = model_path

for name, p in outputs.items():
    print(f'  wrote {name}  ({p.stat().st_size:,} bytes)')

  wrote aim2_signature_primary_predictions.csv  (688 bytes)
  wrote aim2_signature_primary_genes.csv  (29,056 bytes)
  wrote aim2_signature_primary_model.joblib  (21,505 bytes)
  wrote aim2_signature_pragmatic_predictions.csv  (1,044 bytes)
  wrote aim2_signature_pragmatic_genes.csv  (23,475 bytes)
  wrote aim2_signature_pragmatic_model.joblib  (21,505 bytes)


## 10 — Write `data/processed/notebook_11_report.md`

The report is written **without a wall-clock timestamp** so it is byte-identical across runs (the sha256 idempotence check covers it).

In [12]:
def fmt_ci(c):
    return f"[{c[0]:.4f}, {c[1]:.4f}]"

L = []
L.append('# Notebook 11 — Aim 2 ALT Signature Score (DepMap OS RNA-seq)\n\n')
L.append('Regularized ALT signature score on DepMap_26Q1 osteosarcoma RNA-seq using '
         'direction-aware labels (`alt_call_direction`), evaluated by leave-one-out '
         'cross-validation per `label_spec_v1.0` §2. Signature score, **not** a '
         'calibrated classifier and **not** a fixed-threshold predictor.\n\n')

L.append('## Configuration\n')
L.append(f'- Model: `LogisticRegression(penalty="elasticnet", solver="saga", '
         f'l1_ratio={L1_RATIO}, max_iter=10000, class_weight="balanced", '
         f'random_state={RANDOM_SEED})` in `Pipeline(VarianceGeneFilter -> StandardScaler '
         f'-> LogisticRegression)`.\n')
L.append(f'- Fold-local gene-variance filter keeps top `{TOP_K_VARIABLE_GENES}` '
         f'high-variance genes (training-fold variances only); see notebook \u00a76 / '
         f'`src.aim2_signature` for rationale.\n')
L.append(f'- Inner 5-fold CV selects `C` from `10**linspace(-2,2,9)` = '
         f'{[round(float(c),4) for c in C_GRID]}.\n')
L.append(f'- Bootstrap resamples: {N_BOOTSTRAP}; permutation shuffles: {N_PERMUTATIONS}; '
         f'seed: {RANDOM_SEED}.\n\n')

L.append('## Filter trace\n')
for defn in DEFINITIONS:
    L.append(f'### {defn.capitalize()} (admitted alt_status: '
             f'{PRIMARY_ALT_STATUSES if defn=="primary" else PRAGMATIC_ALT_STATUSES})\n')
    L.append('```\n')
    for label, n in traces[defn].steps:
        L.append(f'{label:<55} N = {n}\n')
    L.append(f'direction counts: {direction_counts(filtered[defn])}\n')
    L.append('```\n')
L.append('Count assertions (expected primary 12/8, pragmatic 18/13): **PASS**.\n')
L.append('Ground-truth: ACH-000364 (U2OS) = positive, ACH-000748 (SJSA1) = negative: '
         '**PASS**. Arms derived from `alt_call_direction`, never `alt_status`.\n\n')

L.append('## Cross-validated performance\n')
L.append('| Definition | N (pos/neg) | AUROC [95% CI] | AUPRC [95% CI] | '
         'Sens@90%Spec [95% CI] | Permutation p (AUROC) |\n')
L.append('|---|---|---|---|---|---|\n')
for defn in DEFINITIONS:
    res, m, c, p = loocv[defn], metrics[defn], cis[defn], perms[defn]
    npos, nneg = int(res.y_true.sum()), int((1 - res.y_true).sum())
    L.append(f"| {defn} | {len(res.y_true)} ({npos}/{nneg}) | "
             f"{m['auroc']:.4f} {fmt_ci(c['auroc'])} | "
             f"{m['auprc']:.4f} {fmt_ci(c['auprc'])} | "
             f"{m['sensitivity_at_90_spec']:.4f} {fmt_ci(c['sensitivity_at_90_spec'])} | "
             f"{p['p_value']:.4f} |\n")
L.append('\n')
for defn in DEFINITIONS:
    p = perms[defn]
    L.append(f'- {defn} permutation null: mean AUROC={p["perm_aurocs"].mean():.4f}, '
             f'95th pct={float(np.quantile(p["perm_aurocs"],0.95)):.4f}, '
             f'fixed C={p["permutation_fixed_C"]:.4g} (median observed), '
             f'n={p["n_perm"]}.\n')
L.append('\n')

L.append('## Top-20 genes by |coefficient| (signed; refit on full N at median C)\n')
for defn in DEFINITIONS:
    L.append(f'### {defn.capitalize()}\n')
    L.append('| rank | gene | coefficient | in ALT gene list |\n')
    L.append('|---|---|---:|---|\n')
    for _, r in tops[defn].iterrows():
        L.append(f"| {int(r['rank'])} | {r['gene_symbol']} | {r['coefficient']:+.4f} | "
                 f"{'yes' if r['in_alt_gene_list'] else ''} |\n")
    hits = tops[defn].loc[tops[defn]['in_alt_gene_list'], 'gene_symbol'].tolist()
    L.append(f'\nALT-gene-list hits in top-20: '
             f'{", ".join(hits) if hits else "none"}.\n\n')

L.append('## ALT gene-list cross-check (full ranked list)\n')
L.append(f'Curated ALT-related genes: {", ".join(ALT_GENE_LIST)}.\n\n')
for defn in DEFINITIONS:
    rk = ranked[defn]
    present = rk[rk['gene_symbol'].isin(ALT_GENE_LIST) & (rk['coefficient'] != 0)]
    L.append(f'- {defn}: {len(present)} ALT-list gene(s) with non-zero coefficient')
    if len(present):
        items = ', '.join(f"{r.gene_symbol}(rank {int(r['rank'])}, {r.coefficient:+.4f})"
                          for _, r in present.iterrows())
        L.append(f': {items}')
    L.append('.\n')
L.append('\n')

L.append('## Primary vs pragmatic comparison\n')
L.append(f"- AUROC delta (primary - pragmatic): "
         f"{comparison['auroc_delta_primary_minus_pragmatic']:+.4f}\n")
L.append(f"- Top-20 gene overlap: {comparison['top_overlap_count']} "
         f"(Jaccard {comparison['top_overlap_jaccard']:.3f}); "
         f"genes: {', '.join(comparison['top_overlap_genes']) if comparison['top_overlap_genes'] else 'none'}\n")
L.append(f"- Coefficient Pearson r over {comparison['n_shared_genes']} shared genes: "
         f"{comparison['coef_pearson_on_shared_genes']:.4f}\n\n")

L.append('## Methods notes & limitations\n')
L.append('1. Signature score by LOOCV (label_spec_v1.0 \u00a72): no calibrated '
         'probabilities, no fixed thresholds.\n')
L.append('2. All preprocessing (variance filter, scaler) is fit inside the CV training '
         'fold only; nested inner CV uses its own per-fold scaler.\n')
L.append('3. The gene-variance filter is implemented as a fold-local top-K high-variance '
         f'selector (K={TOP_K_VARIABLE_GENES}) for tractability over ~19k genes at N=20-31.\n')
L.append('4. The permutation null fixes C to the median observed value (re-selecting C in '
         'every fold of every shuffle is intractable); a documented, slightly conservative '
         'concession.\n')
L.append('5. DepMap_26Q1 only; not applied to other cohorts (notebook 12) and never pooled '
         'with de Nonneville microarray.\n')
L.append('6. Very small N (20 strict / 31 pragmatic): treat all estimates as exploratory; '
         'the permutation p-value is the primary honesty check.\n')

report_text = ''.join(L)
REPORT_PATH = PROC / 'notebook_11_report.md'
REPORT_PATH.write_text(report_text)
outputs[REPORT_PATH.name] = REPORT_PATH
print(f'Wrote {REPORT_PATH.relative_to(ROOT)}  ({REPORT_PATH.stat().st_size} bytes)')

Wrote data/processed/notebook_11_report.md  (5564 bytes)


## 11 — Idempotence check (sha256, byte-identical)

Records the sha256 of every output file. On re-run the freshly written files must match
the recorded hashes exactly (deterministic seeds + CV iterators). A mismatch raises
`AssertionError`. None of the hashed outputs embeds a wall-clock timestamp.

In [13]:
import json

SHA_PATH = PROC / 'notebook_11_sha256.json'
HASHED = [
    'aim2_signature_primary_predictions.csv',
    'aim2_signature_pragmatic_predictions.csv',
    'aim2_signature_primary_genes.csv',
    'aim2_signature_pragmatic_genes.csv',
    'aim2_signature_primary_model.joblib',
    'aim2_signature_pragmatic_model.joblib',
    'notebook_11_report.md',
]
current = {name: sha256_file(PROC / name) for name in HASHED}
for name in HASHED:
    print(f'  {name}: {current[name]}')

if SHA_PATH.exists():
    previous = json.loads(SHA_PATH.read_text())
    mismatch = {k: (previous.get(k), current[k]) for k in HASHED if previous.get(k) != current[k]}
    if mismatch:
        print('\nIDEMPOTENCE FAILED — outputs differ from recorded sha256:')
        for k, (was, now) in mismatch.items():
            print(f'  {k}: was {was} now {now}')
        raise AssertionError(f'Idempotence FAILED for: {sorted(mismatch)}')
    print('\nIdempotence check PASSED: all output files byte-identical to recorded sha256.')
else:
    print('\nFirst run: recorded sha256 baseline (re-run to verify byte-identity).')

SHA_PATH.write_text(json.dumps(current, indent=2, sort_keys=True) + '\n')
print(f'sha256 manifest: {SHA_PATH.relative_to(ROOT)}')

  aim2_signature_primary_predictions.csv: e10b32b4648f9c5b3368233e6d5478f615a3e8cf85e3222007b3269fbfa4bf01
  aim2_signature_pragmatic_predictions.csv: 638f4881eabfebf0b04282cca4ad7d2b96e395b49b264a8f2094326330c31524
  aim2_signature_primary_genes.csv: 8c6044a292c431e14ccb6a3770f7a6d2fc6f0e52437c16d0e01563bdb950502b
  aim2_signature_pragmatic_genes.csv: 3ca9f60e4d5a6e70ba9417c156ff3f9bba4af680b324f77cb715db310563e567
  aim2_signature_primary_model.joblib: c0230888c25bb235c8b4d0b0c794084012a3e298521a759aa1bc15b7b7d03256
  aim2_signature_pragmatic_model.joblib: 17faaadc41e40a24a3a9dd04e50e188c83976e796c288a1faaa0ddbadfe643cb
  notebook_11_report.md: ce835cb077bebed5d075734749fe9c431c03b9a6e55464a173684cf51e793fa4

Idempotence check PASSED: all output files byte-identical to recorded sha256.
sha256 manifest: data/processed/notebook_11_sha256.json
